### Load the dataset

In [71]:
import pandas as pd

df = pd.read_csv("../data/crm/telco_churn.csv")
df.head()


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


### 1.EDA:

In this section, I inspect the structure of the CRM dataset, check data types, missing values, and the distribution of the target variable (Churn). This helps me understand data quality and identify issues that need cleaning.

### 1.1.Check the shape

In [73]:
df.shape


(7043, 21)

### 1.2.Check column names

In [75]:
df.columns


Index(['customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents',
       'tenure', 'PhoneService', 'MultipleLines', 'InternetService',
       'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport',
       'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling',
       'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn'],
      dtype='object')

### 1.3.Check data types

In [77]:
df.dtypes


customerID           object
gender               object
SeniorCitizen         int64
Partner              object
Dependents           object
tenure                int64
PhoneService         object
MultipleLines        object
InternetService      object
OnlineSecurity       object
OnlineBackup         object
DeviceProtection     object
TechSupport          object
StreamingTV          object
StreamingMovies      object
Contract             object
PaperlessBilling     object
PaymentMethod        object
MonthlyCharges      float64
TotalCharges         object
Churn                object
dtype: object

### 1.4.Check missing values

In [79]:
df.isnull().sum()


customerID          0
gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            0
PaperlessBilling    0
PaymentMethod       0
MonthlyCharges      0
TotalCharges        0
Churn               0
dtype: int64

### 1.6.Check target variable distribution

In [81]:
df['Churn'].value_counts(normalize=True)


Churn
No     0.73463
Yes    0.26537
Name: proportion, dtype: float64

### 1.7.Basic statistics

In [83]:
df.describe()


,SeniorCitizen,tenure,MonthlyCharges
count,7043.000000,7043.000000,7043.000000
mean,0.162147,32.371149,64.761692
std,0.368612,24.559481,30.090047
min,0.000000,0.000000,18.250000
25%,0.000000,9.000000,35.500000
50%,0.000000,29.000000,70.350000
75%,0.000000,55.000000,89.850000
max,1.000000,72.000000,118.750000


## report: 
The CRM dataset needs cleaning because the TotalCharges column is incorrectly stored as text and contains non-numeric values. Many columns are categorical and must be encoded before modeling. The dataset is also imbalanced, which affects model training. Cleaning ensures the data is consistent, numeric where required, and ready for machine learning.

### 2.Data Cleaning

In this step, I clean the CRM dataset by converting the TotalCharges column to numeric, handling missing values, and encoding categorical variables. This ensures the dataset is in a suitable format for machine learning models.

### 2.1. Fixing TotalCharges 
Convert it to numeric, 
Then check missing values again,
Remove rows where TotalCharges is missing,
Convert target variable to numeric,
Verify everything is numeric

In [85]:
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')


In [87]:
df['TotalCharges'].isnull().sum()


11

In [89]:
df = df.dropna(subset=['TotalCharges'])


In [91]:
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})


In [95]:
df.dtypes


SeniorCitizen                              int64
tenure                                     int64
MonthlyCharges                           float64
TotalCharges                             float64
Churn                                      int64
                                          ...   
Contract_Two year                           bool
PaperlessBilling_Yes                        bool
PaymentMethod_Credit card (automatic)       bool
PaymentMethod_Electronic check              bool
PaymentMethod_Mailed check                  bool
Length: 7062, dtype: object

### 2.2 Encoding categorical variables

In [93]:
df = pd.get_dummies(df, drop_first=True)


### 2.3 Handling imbalance

In [67]:
class_weight='balanced'


### 3.Train/Test Split and Baseline Model
Train the model on one portion

Test it on unseen data

Measure accuracy, precision, recall, F1
In this step, I split the cleaned CRM dataset into training and testing sets using a stratified split to preserve the churn distribution. I then trained a baseline Logistic Regression model with class weighting to handle imbalance. I evaluated the model using accuracy, precision, recall, F1 score, and a confusion matrix.
I split the dataset into training and testing sets using a stratified split to preserve the churn distribution.

I trained a Logistic Regression model with class weighting to handle imbalance.

I generated predictions on the test set and evaluated the model using precision, recall, F1-score, and a confusion matrix.

Recall for the churn class is the most important metric because the goal is to identify customers likely to leave.


### 3.1. Train/Test Split

In [100]:
from sklearn.model_selection import train_test_split

X = df.drop('Churn', axis=1)
y = df['Churn']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


### 3.2.Train the Logistic Regression Model

In [105]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000, class_weight='balanced')
model.fit(X_train, y_train)


C:\Users\Mahsa\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


LogisticRegression(class_weight='balanced', max_iter=1000)

### 3.3. Make prediction

In [109]:
y_pred = model.predict(X_test)


### 3.4. Evaluate the Model


In [111]:
from sklearn.metrics import classification_report, confusion_matrix

print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))


              precision    recall  f1-score   support

           0       0.90      0.70      0.79      1033
           1       0.49      0.80      0.61       374

    accuracy                           0.72      1407
   macro avg       0.70      0.75      0.70      1407
weighted avg       0.79      0.72      0.74      1407

[[720 313]
 [ 76 298]]


### 4.Train a Random Forest model

I trained a Random Forest classifier to improve churn prediction beyond the baseline Logistic Regression model. Random Forest handles non-linear relationships, interactions between features, and imbalanced data more effectively. I evaluated the model using precision, recall, F1-score, and a confusion matrix. I also examined feature importance to understand which factors most influence churn.

### 4.1. Import and Train the Model

In [116]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    class_weight='balanced',
    random_state=42
)

rf_model.fit(X_train, y_train)


RandomForestClassifier(class_weight='balanced', n_estimators=300,
                       random_state=42)

### 4.2. Make Predictions

In [118]:
rf_pred = rf_model.predict(X_test)


### 4.3. Evaluate the Model

In [120]:
from sklearn.metrics import classification_report, confusion_matrix

print(classification_report(y_test, rf_pred))
print(confusion_matrix(y_test, rf_pred))


              precision    recall  f1-score   support

           0       0.83      0.89      0.86      1033
           1       0.63      0.51      0.56       374

    accuracy                           0.79      1407
   macro avg       0.73      0.70      0.71      1407
weighted avg       0.78      0.79      0.78      1407

[[923 110]
 [185 189]]


### 4.4. Feature Importance

In [122]:
import pandas as pd
import numpy as np

importances = rf_model.feature_importances_
indices = np.argsort(importances)[::-1]

feature_names = X_train.columns

for i in range(10):
    print(f"{feature_names[indices[i]]}: {importances[indices[i]]}")


TotalCharges: 0.09713926637716658
tenure: 0.09598076004082273
MonthlyCharges: 0.07975989824392643
Contract_Two year: 0.03733310304392299
InternetService_Fiber optic: 0.03083000046804442
PaymentMethod_Electronic check: 0.025917530880076322
OnlineSecurity_Yes: 0.02365222609640743
Contract_One year: 0.022612426230358064
TechSupport_Yes: 0.021567355286518604
PaperlessBilling_Yes: 0.01848066671528985


### 5.Train an XGBoost model

In [124]:
pip install xgboost


   ---------------------------------------- 0.0/101.7 MB ? eta -:--:--
   ---------------------------------------- 0.5/101.7 MB 4.2 MB/s eta 0:00:25
    --------------------------------------- 1.3/101.7 MB 3.7 MB/s eta 0:00:27
    --------------------------------------- 2.4/101.7 MB 3.9 MB/s eta 0:00:26
   - -------------------------------------- 3.1/101.7 MB 3.9 MB/s eta 0:00:26
   - -------------------------------------- 3.9/101.7 MB 4.0 MB/s eta 0:00:25
   - -------------------------------------- 4.7/101.7 MB 4.0 MB/s eta 0:00:25
   -- ------------------------------------- 5.5/101.7 MB 3.9 MB/s eta 0:00:25
   -- ------------------------------------- 6.8/101.7 MB 4.2 MB/s eta 0:00:23
   --- ------------------------------------ 8.1/101.7 MB 4.3 MB/s eta 0:00:22
   --- ------------------------------------ 9.4/101.7 MB 4.6 MB/s eta 0:00:21
   ---- ----------------------------------- 10.7/101.7 MB 4.7 MB/s eta 0:00:20
   ---- ----------------------------------- 12.1/101.7 MB 4.8 MB/s eta

### 5.1. Import and Train the Model

In [126]:
from xgboost import XGBClassifier

xgb_model = XGBClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=(len(y_train[y_train==0]) / len(y_train[y_train==1])),
    random_state=42,
    eval_metric='logloss'
)

xgb_model.fit(X_train, y_train)


XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.05, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=6, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=300, n_jobs=None,
              num_parallel_tree=None, ...)

### 5.2. Make Predictions

In [128]:
xgb_pred = xgb_model.predict(X_test)


### 5.3. Evaluate the Model

In [130]:
from sklearn.metrics import classification_report, confusion_matrix

print(classification_report(y_test, xgb_pred))
print(confusion_matrix(y_test, xgb_pred))


              precision    recall  f1-score   support

           0       0.88      0.76      0.82      1033
           1       0.53      0.72      0.61       374

    accuracy                           0.75      1407
   macro avg       0.71      0.74      0.71      1407
weighted avg       0.79      0.75      0.76      1407

[[789 244]
 [103 271]]


### 5.4. Feature Importance

In [132]:
import pandas as pd
import numpy as np

importances = xgb_model.feature_importances_
indices = np.argsort(importances)[::-1]

feature_names = X_train.columns

for i in range(10):
    print(f"{feature_names[indices[i]]}: {importances[indices[i]]}")


Contract_Two year: 0.32802650332450867
InternetService_Fiber optic: 0.15510618686676025
Contract_One year: 0.09854385256767273
InternetService_No: 0.069204181432724
OnlineSecurity_No internet service: 0.03773760423064232
OnlineBackup_No internet service: 0.029295852407813072
StreamingMovies_Yes: 0.025074666365981102
tenure: 0.021091114729642868
PaymentMethod_Electronic check: 0.019062193110585213
PhoneService_Yes: 0.01712990179657936


### 6.Build a TensorFlow neural network

In [134]:
pip install tensorflow


Note: you may need to restart the kernel to use updated packages.


In [136]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers


### 6.1. Prepare data for TensorFlow

Neural networks work better with scaled features.

In [138]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_tf = scaler.fit_transform(X_train)
X_test_tf = scaler.transform(X_test)


### 6.2. Build the neural network

In [141]:
input_dim = X_train_tf.shape[1]

model_tf = keras.Sequential([
    layers.Input(shape=(input_dim,)),
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(32, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(1, activation='sigmoid')
])

model_tf.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model_tf.summary()


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 64)             │       451,968 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 454,081 (1.73 MB)

 Trainable params: 454,081 (1.73 MB)

 Non-trainable params: 0 (0.00 B)

### 6.3. Train the model

In [144]:
history = model_tf.fit(
    X_train_tf,
    y_train,
    validation_split=0.2,
    epochs=30,
    batch_size=64,
    verbose=1
)


Epoch 1/30
71/71 ━━━━━━━━━━━━━━━━━━━━ 4s 22ms/step - accuracy: 0.6967 - loss: 0.6190 - val_accuracy: 0.7449 - val_loss: 0.5128
Epoch 2/30
71/71 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.7976 - loss: 0.4188 - val_accuracy: 0.7493 - val_loss: 0.4935
Epoch 3/30
71/71 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - accuracy: 0.9053 - loss: 0.2301 - val_accuracy: 0.7360 - val_loss: 0.5109
Epoch 4/30
71/71 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.9673 - loss: 0.1005 - val_accuracy: 0.7360 - val_loss: 0.5471
Epoch 5/30
71/71 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.9887 - loss: 0.0447 - val_accuracy: 0.7289 - val_loss: 0.6183
Epoch 6/30
71/71 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - accuracy: 0.9960 - loss: 0.0223 - val_accuracy: 0.7218 - val_loss: 0.6713
Epoch 7/30
71/71 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.9964 - loss: 0.0136 - val_accuracy: 0.7156 - val_loss: 0.7047
Epoch 8/30
71/71 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.9991 - loss: 0.0090 - val_accuracy: 0.7031 - v

### 6.4. Evaluate on test set

In [147]:
y_pred_proba_tf = model_tf.predict(X_test_tf)
y_pred_tf = (y_pred_proba_tf >= 0.5).astype(int)

from sklearn.metrics import classification_report, confusion_matrix

print(classification_report(y_test, y_pred_tf))
print(confusion_matrix(y_test, y_pred_tf))


44/44 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step
              precision    recall  f1-score   support

           0       0.79      0.95      0.87      1033
           1       0.71      0.31      0.43       374

    accuracy                           0.78      1407
   macro avg       0.75      0.63      0.65      1407
weighted avg       0.77      0.78      0.75      1407

[[986  47]
 [259 115]]


### 7.Build a PyTorch neural network

In [167]:
pip install torch torchvision torchaudio


   ---------------------------------------- 0.0/4.0 MB ? eta -:--:--
   ------- -------------------------------- 0.8/4.0 MB 4.2 MB/s eta 0:00:01
   --------------- ------------------------ 1.6/4.0 MB 5.2 MB/s eta 0:00:01
   -------------------------- ------------- 2.6/4.0 MB 4.7 MB/s eta 0:00:01
   ---------------------------------------  3.9/4.0 MB 5.0 MB/s eta 0:00:01
   ---------------------------------------- 4.0/4.0 MB 4.7 MB/s eta 0:00:00
   ---------------------------------------- 0.0/123.0 MB ? eta -:--:--
   ---------------------------------------- 1.0/123.0 MB 5.0 MB/s eta 0:00:25
    --------------------------------------- 1.6/123.0 MB 4.2 MB/s eta 0:00:29
    --------------------------------------- 2.9/123.0 MB 4.8 MB/s eta 0:00:26
   - -------------------------------------- 3.9/123.0 MB 4.7 MB/s eta 0:00:26
   - -------------------------------------- 5.2/123.0 MB 5.0 MB/s eta 0:00:24
   -- ------------------------------------- 6.3/123.0 MB 5.1 MB/s eta 0:00:23
   -- ------

In [172]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader


### 7.1.Prepare data for PyTorch

In [174]:
# Convert to tensors
X_train_torch = torch.tensor(X_train_tf, dtype=torch.float32)
X_test_torch = torch.tensor(X_test_tf, dtype=torch.float32)
y_train_torch = torch.tensor(y_train.values, dtype=torch.float32).view(-1, 1)
y_test_torch = torch.tensor(y_test.values, dtype=torch.float32).view(-1, 1)

# Create datasets
train_dataset = TensorDataset(X_train_torch, y_train_torch)
test_dataset = TensorDataset(X_test_torch, y_test_torch)

# Create dataloaders
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)


### 7.2. Build the PyTorch neural network

In [177]:
class ChurnNet(nn.Module):
    def __init__(self, input_dim):
        super(ChurnNet, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(32, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.model(x)

input_dim = X_train_torch.shape[1]
model_pt = ChurnNet(input_dim)


### 7.3. Define loss and optimizer

In [180]:
criterion = nn.BCELoss()
optimizer = optim.Adam(model_pt.parameters(), lr=0.001)


### 7.4. Train the PyTorch model

In [183]:
epochs = 30

for epoch in range(epochs):
    model_pt.train()
    running_loss = 0.0

    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        outputs = model_pt(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    print(f"Epoch {epoch+1}/{epochs}, Loss: {running_loss/len(train_loader):.4f}")


Epoch 1/30, Loss: 0.5487
Epoch 2/30, Loss: 0.6548
Epoch 3/30, Loss: 0.3075
Epoch 4/30, Loss: 0.1240
Epoch 5/30, Loss: 0.0452
Epoch 6/30, Loss: 0.0253
Epoch 7/30, Loss: 0.0174
Epoch 8/30, Loss: 0.0114
Epoch 9/30, Loss: 0.0114
Epoch 10/30, Loss: 0.0105
Epoch 11/30, Loss: 0.0082
Epoch 12/30, Loss: 0.0063
Epoch 13/30, Loss: 0.0081
Epoch 14/30, Loss: 0.0064
Epoch 15/30, Loss: 0.0053
Epoch 16/30, Loss: 0.0047
Epoch 17/30, Loss: 0.0047
Epoch 18/30, Loss: 0.0051
Epoch 19/30, Loss: 0.0038
Epoch 20/30, Loss: 0.0055
Epoch 21/30, Loss: 0.0045
Epoch 22/30, Loss: 0.0039
Epoch 23/30, Loss: 0.0029
Epoch 24/30, Loss: 0.0036
Epoch 25/30, Loss: 0.0032
Epoch 26/30, Loss: 0.0032
Epoch 27/30, Loss: 0.0031
Epoch 28/30, Loss: 0.0031
Epoch 29/30, Loss: 0.0038
Epoch 30/30, Loss: 0.0027


#### 7.5 Evaluate the model

In [186]:
model_pt.eval()
with torch.no_grad():
    y_pred_proba_pt = model_pt(X_test_torch).numpy()
    y_pred_pt = (y_pred_proba_pt >= 0.5).astype(int)


### 7.6. Classification report

In [188]:
from sklearn.metrics import classification_report, confusion_matrix

print(classification_report(y_test, y_pred_pt))
print(confusion_matrix(y_test, y_pred_pt))


              precision    recall  f1-score   support

           0       0.73      1.00      0.85      1033
           1       0.00      0.00      0.00       374

    accuracy                           0.73      1407
   macro avg       0.37      0.50      0.42      1407
weighted avg       0.54      0.73      0.62      1407

[[1033    0]
 [ 374    0]]


C:\Users\Mahsa\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\Mahsa\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\Mahsa\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


### 8.Compare all models

| Model | Recall (Churn) | Precision (Churn) | F1 (Churn) | Accuracy | Notes |
| --- | --- | --- | --- | --- | --- |
| **[Logistic Regression](ca://s?q=Explain_Logistic_Regression_results)** | **0.80** | 0.49 | 0.61 | 0.72 | Best recall, simple, interpretable |
| **[Random Forest](ca://s?q=Explain_Random_Forest_results)** | 0.51 | **0.63** | 0.56 | **0.79** | Best precision, best accuracy |
| **[XGBoost](ca://s?q=Explain_XGBoost_results)** | 0.72 | 0.58 | **0.64** | 0.74 | Best F1, best balance overall |
| **[TensorFlow NN](ca://s?q=Explain_TensorFlow_results)** | 0.31 | 0.71 | 0.43 | 0.78 | Overfitting, weak recall |
| **[PyTorch NN](ca://s?q=Explain_PyTorch_results)** | 0.00 | 0.00 | 0.00 | 0.73 | Collapsed to predicting all “No churn” |

1. Best model for business value → XGBoost
Because churn prediction cares most about:

catching churners (recall)

while keeping predictions reliable (precision)

and balancing both (F1-score)

XGBoost wins with:

Recall = 0.72

Precision = 0.58

F1 = 0.64 (highest)

Accuracy = 0.74

This is the most balanced and strongest performer.

2. Best model for interpretability → Logistic Regression
If a manager wants:

“Which features increase churn?”

“How much does tenure reduce churn?”

Logistic Regression is the best storytelling model.

3. Best model for accuracy → Random Forest
Random Forest gives:

Accuracy = 0.79 (highest)

But accuracy is misleading in imbalanced datasets.

4. Deep learning models underperformed
Both TensorFlow and PyTorch:

Overfit

Struggled with imbalance

Performed worse than XGBoost

This is normal for small tabular datasets.

### 9.Save the best model

In [249]:
df = pd.read_csv("../data/crm/telco_churn.csv")



In [251]:
df.columns

Index(['customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents',
       'tenure', 'PhoneService', 'MultipleLines', 'InternetService',
       'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport',
       'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling',
       'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn'],
      dtype='object')

In [253]:
##Convert TotalCharges to numeric
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
df = df.dropna(subset=["TotalCharges"])


In [255]:
df = df.drop(columns=["customerID"])


In [257]:
df.columns

Index(['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure',
       'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity',
       'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV',
       'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod',
       'MonthlyCharges', 'TotalCharges', 'Churn'],
      dtype='object')

In [259]:
df = pd.get_dummies(df, drop_first=True)



In [261]:
df.columns

Index(['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges',
       'gender_Male', 'Partner_Yes', 'Dependents_Yes', 'PhoneService_Yes',
       'MultipleLines_No phone service', 'MultipleLines_Yes',
       'InternetService_Fiber optic', 'InternetService_No',
       'OnlineSecurity_No internet service', 'OnlineSecurity_Yes',
       'OnlineBackup_No internet service', 'OnlineBackup_Yes',
       'DeviceProtection_No internet service', 'DeviceProtection_Yes',
       'TechSupport_No internet service', 'TechSupport_Yes',
       'StreamingTV_No internet service', 'StreamingTV_Yes',
       'StreamingMovies_No internet service', 'StreamingMovies_Yes',
       'Contract_One year', 'Contract_Two year', 'PaperlessBilling_Yes',
       'PaymentMethod_Credit card (automatic)',
       'PaymentMethod_Electronic check', 'PaymentMethod_Mailed check',
       'Churn_Yes'],
      dtype='object')

In [265]:
## Split into X and y
X = df.drop("Churn_Yes", axis=1)
y = df["Churn_Yes"]


In [267]:
## Train/test split
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


In [271]:
## Save the correct feature names
import json

feature_names = list(X_train.columns)

with open("feature_names.json", "w") as f:
    json.dump(feature_names, f)


In [279]:
## Retrain  XGBoost model once
xgb_model.fit(X_train, y_train)


XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.05, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=6, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=300, n_jobs=None,
              num_parallel_tree=None, ...)

In [281]:
## save the file 
joblib.dump(xgb_model, "xgb_churn_model.pkl")


['xgb_churn_model.pkl']

### 8. Model Comparison

Below is a comparison of all machine learning and deep learning models trained on the CRM churn dataset. This table highlights accuracy, recall, precision, and F1-score for the churn class (1), which is the most important metric for identifying customers likely to leave.



Logistic Regression	scikit‑learn	0.72	0.80	0.49	0.61
Random Forest	scikit‑learn	0.79	0.51	0.63	0.56
XGBoost	scikit‑learn / XGBoost	0.75	0.72	0.53	0.61
Neural Network	TensorFlow	0.78	0.31	0.71	0.43
Neural Network	PyTorch	0.73	0.00	0.00	0.00

### Interpretation

- Random Forest achieved the highest overall accuracy (0.79).
- Logistic Regression achieved the best recall for churners (0.80), which is critical for churn prediction.
- XGBoost provided a strong balance between recall and precision.
- TensorFlow and PyTorch models demonstrated deep learning capability, but struggled with recall due to dataset size and imbalance.


In [287]:
import os
os.makedirs("models", exist_ok=True)


In [289]:
import joblib
joblib.dump(model, "models/logistic_regression.pkl")


['models/logistic_regression.pkl']

In [291]:
joblib.dump(rf_model, "models/random_forest.pkl")


['models/random_forest.pkl']

In [293]:
joblib.dump(xgb_model, "models/xgboost_model.pkl")


['models/xgboost_model.pkl']

In [295]:
model_tf.save("models/tensorflow_model.h5")


In [297]:
import torch
torch.save(model_pt.state_dict(), "models/pytorch_model.pt")


In [299]:
os.listdir("models")


['logistic_regression.pkl',
 'pytorch_model.pt',
 'random_forest.pkl',
 'tensorflow_model.h5',
 'xgboost_model.pkl']